In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

def load_data(train_path, test_path):
    train_df=pd.read_csv(train_path)
    test_df=pd.read_csv(test_path)

    print("Train shape:",train_df.shape)
    print("Test shape :",test_df.shape)
    print("Missing values in train:",train_df.isna().sum().sum())
    print("Missing values in test :",test_df.isna().sum().sum())

    X_train=train_df.drop("Class",axis=1)
    y_train=train_df["Class"]
    X_test=test_df.drop("ID",axis=1)
    test_ids=test_df["ID"]

    return X_train,y_train,X_test,test_ids


print("Loading data")
train_path = r"C:\Users\komal\Documents\alrIEEEna_26_dataset\MLChallengeDataset\TRAIN.csv"
test_path = r"C:\Users\komal\Documents\alrIEEEna_26_dataset\MLChallengeDataset\TEST.csv"
X_train_raw, y_train, X_test_raw, test_ids = load_data(train_path, test_path)

Loading data...
Train shape: (43776, 48)
Test shape : (10944, 48)
Missing values in train: 0
Missing values in test : 0


In [3]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier

def handle_outliers(X_train, X_test):
    print("Checking outliers")
    train_cap = X_train.copy()
    test_cap = X_test.copy()
    total_outliers = 0
    feature_cols = [c for c in X_train.columns if c.startswith("F")]

    for col in feature_cols:
        q1 = X_train[col].quantile(0.25)
        q3 = X_train[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
        total_outliers += outliers
        low_cap = X_train[col].quantile(0.01)
        high_cap = X_train[col].quantile(0.99)
        train_cap[col] = np.clip(X_train[col], low_cap, high_cap)
        test_cap[col] = np.clip(X_test[col], low_cap, high_cap)

    print("Total outliers detected:", total_outliers)
    return train_cap, test_cap

X_train_capped, X_test_capped = handle_outliers(X_train_raw, X_test_raw)

Checking outliers
Total outliers detected: 272392


In [6]:
def feature_engineering(df):
    df_copy = df.copy()
    sensor_cols = [c for c in df_copy.columns if c.startswith("F")]
    new_features = {}

    new_features["sensor_mean"] = df_copy[sensor_cols].mean(axis=1)
    new_features["sensor_std"] = df_copy[sensor_cols].std(axis=1)
    new_features["sensor_skew"] = df_copy[sensor_cols].skew(axis=1)
    new_features["sensor_kurt"] = df_copy[sensor_cols].kurtosis(axis=1)

    new_features["sensor_min"] = df_copy[sensor_cols].min(axis=1)
    new_features["sensor_max"] = df_copy[sensor_cols].max(axis=1)

    new_features["sensor_median"] = df_copy[sensor_cols].median(axis=1)
    new_features["sensor_q25"] = df_copy[sensor_cols].quantile(0.25, axis=1)
    new_features["sensor_q75"] = df_copy[sensor_cols].quantile(0.75, axis=1)

    for i in range(1, len(sensor_cols)):
        prev_col = sensor_cols[i - 1]
        curr_col = sensor_cols[i]
        roc_name = f"roc_{prev_col}_{curr_col}"
        new_features[roc_name] = df_copy[curr_col] - df_copy[prev_col]

    for col in sensor_cols:
        dev_name = f"dev_{col}"
        new_features[dev_name] = df_copy[col] - new_features["sensor_mean"]

    new_df = pd.DataFrame(new_features)
    result = pd.concat([df_copy, new_df], axis=1)
    return result

X_train_eng = feature_engineering(X_train_capped)
X_test_eng = feature_engineering(X_test_capped)
print("complete")

complete


In [7]:
def feature_selection(X_train, y_train, X_test, drop_bottom_pct=0.20):
    print("Feature selection")

    model = LGBMClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X_train, y_train)

    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
    keep_count = int(len(X_train.columns) * (1 - drop_bottom_pct))
    top_features = importance_df.head(keep_count)["feature"].tolist()

    print("Keeping features:",keep_count)
    print("Dropped features:",len(X_train.columns) - keep_count)
    return X_train[top_features].copy(), X_test[top_features].copy()

X_train_clean, X_test_clean = feature_selection(
    X_train_eng,
    y_train,
    X_test_eng,
    drop_bottom_pct=0.15
)

Feature selection
Keeping features: 126
Dropped features: 23


In [8]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [10]:
def try_models(X_train, y_train):
    models = {
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "LogisticRegression": LogisticRegression(max_iter=500, random_state=42),
        "NaiveBayes": GaussianNB(),
        "XGBoost": XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),
        "CatBoost": CatBoostClassifier(
            verbose=0,
            random_state=42,
            thread_count=-1
        ),
        "LightGBM": LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1
        ),
        "HistGradientBoost": HistGradientBoostingClassifier(random_state=42)
    }
    scores = {}
    for name, model in models.items():
        try:
            cv_scores = cross_val_score(
                model,
                X_train,
                y_train,
                cv=3,
                scoring="f1_macro",
                n_jobs=-1
            )
            mean_score = cv_scores.mean()
            scores[name] = mean_score
            print(f"{name}(F1_macro): {mean_score:.4f}")
        except Exception as e:
            print(f"{name} failed:", e)

    sorted_models= sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nTop models:")
    for name,score in sorted_models[:4]:
        print(f"{name}: {score:.4f}")

print("\nModel comparison")
try_models(X_train_clean, y_train)


Model comparison
AdaBoost(F1_macro): 0.7183
GradientBoost(F1_macro): 0.8734
RandomForest(F1_macro): 0.9780
DecisionTree(F1_macro): 0.9358
LogisticRegression(F1_macro): 0.7020
NaiveBayes(F1_macro): 0.6569
XGBoost(F1_macro): 0.9832
CatBoost(F1_macro): 0.9843
LightGBM(F1_macro): 0.9749
HistGradientBoost(F1_macro): 0.9751

Top models:
CatBoost: 0.9843
XGBoost: 0.9832
RandomForest: 0.9780
HistGradientBoost: 0.9751


In [ ]:
import optuna
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score